In [54]:
import torch
from torch.testing import assert_close

In [43]:
def small_tensor_string(
    t, name="", row_limit=50, col_limit=150, min_important_value=1e-5, sci_mode=None, precision=None,
):
    # Unfortunately, pytest usually captures the output and so we can't access the real width here :-(
    # col_limit = col_limit or shutil.get_terminal_size().columns
    shape = "x".join([str(i) for i in t.shape])
    if len(t.shape) > 2:
        t = t.squeeze()
    if len(t.shape) < 2:
        t = t.unsqueeze(0)

    abs = torch.abs(t)
    # Things large enough that we can't round them off to zero and small enough
    # that we need scientific notation to print them or if anything's big enough
    # that we need scientific notation.
    sci_mode = sci_mode if sci_mode is not None else (
        torch.any(torch.logical_and(abs > min_important_value, abs < 1e-3))
        or torch.max(abs) > 1e3
    )
    precision = precision if precision is not None else (2 if sci_mode else 3)

    def fallback():
        with torch._tensor_str.printoptions(
            precision=2 if sci_mode else 3,
            linewidth=col_limit,
            sci_mode=sci_mode,
            threshold=0,
        ):
            return f"{name}[{shape}], {t.dtype}:\n{t}"

    if len(t.shape) > 2 or t.shape[0] > row_limit:
        return fallback()

    def f_entry(d, width=0):
        return f"{d: {width}.{precision}e}" if sci_mode else f"{d: {width}.{precision}f}"

    width = max(len(f_entry(d)) for d in t.flatten().tolist())

    row_width = len(" ".join(f_entry(d, width) for d in t[0].tolist()))

    if row_width > col_limit:
        print(f"{row_width=} > {col_limit=}")
        return fallback()

    rows = []
    for row in t:
        rows.append(" ".join(f_entry(d, width) for d in row.tolist()))

    nl = "\n"
    return f"{name}[{shape}], {t.dtype}:\n{nl.join(rows)}"

In [3]:
def row_str(row):
    return ' '.join(f'{x:3d}' for x in row)

In [4]:
t = torch.arange(256).reshape(16, 16)
for row in t:
    print(row_str(row))

  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31
 32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47
 48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63
 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79
 80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95
 96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111
112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127
128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159
160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175
176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191
192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207
208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223
224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239
240 241 242 243 244 245 246 247 248 249 

In [5]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  has_row = t[i, j:j+4]
  wants_row = t.transpose(0, 1)[i, j:j+4]
  print(f"{x:2d}:   has {row_str(has_row)},   wants {row_str(wants_row)}")

 0:   has   0   1   2   3,   wants   0  16  32  48
 1:   has  16  17  18  19,   wants   1  17  33  49
 2:   has  32  33  34  35,   wants   2  18  34  50
 3:   has  48  49  50  51,   wants   3  19  35  51
 4:   has  64  65  66  67,   wants   4  20  36  52
 5:   has  80  81  82  83,   wants   5  21  37  53
 6:   has  96  97  98  99,   wants   6  22  38  54
 7:   has 112 113 114 115,   wants   7  23  39  55
 8:   has 128 129 130 131,   wants   8  24  40  56
 9:   has 144 145 146 147,   wants   9  25  41  57
10:   has 160 161 162 163,   wants  10  26  42  58
11:   has 176 177 178 179,   wants  11  27  43  59
12:   has 192 193 194 195,   wants  12  28  44  60
13:   has 208 209 210 211,   wants  13  29  45  61
14:   has 224 225 226 227,   wants  14  30  46  62
15:   has 240 241 242 243,   wants  15  31  47  63
16:   has   4   5   6   7,   wants  64  80  96 112
17:   has  20  21  22  23,   wants  65  81  97 113
18:   has  36  37  38  39,   wants  66  82  98 114
19:   has  52  53  54  55,   wa

In [6]:
els_to_thread_and_idx = {}
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  has_row = t[i, j:j+4]
  for v_idx, el in enumerate(has_row.tolist()):
    assert el not in els_to_thread_and_idx
    els_to_thread_and_idx[el] = (x, v_idx)

In [7]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  wants_row = t.transpose(0, 1)[i, j:j+4]
  
  wants_idxs = []
  for el in wants_row.tolist():
    wants_idxs.append(els_to_thread_and_idx[el])

  print(f"{x:2d}:  {' '.join(f'{w[0]:2d}:{w[1]}' for w in wants_idxs)}")

 0:   0:0  1:0  2:0  3:0
 1:   0:1  1:1  2:1  3:1
 2:   0:2  1:2  2:2  3:2
 3:   0:3  1:3  2:3  3:3
 4:  16:0 17:0 18:0 19:0
 5:  16:1 17:1 18:1 19:1
 6:  16:2 17:2 18:2 19:2
 7:  16:3 17:3 18:3 19:3
 8:  32:0 33:0 34:0 35:0
 9:  32:1 33:1 34:1 35:1
10:  32:2 33:2 34:2 35:2
11:  32:3 33:3 34:3 35:3
12:  48:0 49:0 50:0 51:0
13:  48:1 49:1 50:1 51:1
14:  48:2 49:2 50:2 51:2
15:  48:3 49:3 50:3 51:3
16:   4:0  5:0  6:0  7:0
17:   4:1  5:1  6:1  7:1
18:   4:2  5:2  6:2  7:2
19:   4:3  5:3  6:3  7:3
20:  20:0 21:0 22:0 23:0
21:  20:1 21:1 22:1 23:1
22:  20:2 21:2 22:2 23:2
23:  20:3 21:3 22:3 23:3
24:  36:0 37:0 38:0 39:0
25:  36:1 37:1 38:1 39:1
26:  36:2 37:2 38:2 39:2
27:  36:3 37:3 38:3 39:3
28:  52:0 53:0 54:0 55:0
29:  52:1 53:1 54:1 55:1
30:  52:2 53:2 54:2 55:2
31:  52:3 53:3 54:3 55:3
32:   8:0  9:0 10:0 11:0
33:   8:1  9:1 10:1 11:1
34:   8:2  9:2 10:2 11:2
35:   8:3  9:3 10:3 11:3
36:  24:0 25:0 26:0 27:0
37:  24:1 25:1 26:1 27:1
38:  24:2 25:2 26:2 27:2
39:  24:3 25:3 26:3 27:3


In [8]:
for x in range(64):
  i = x % 16
  j = (x // 16)*4
  wants_row = t.transpose(0, 1)[i, j:j+4]
  
  wants_el = wants_row.tolist()[0]
  wants_t, wants_v_idx = els_to_thread_and_idx[wants_el]
  row = (x // 16)
  col = (x % 16)
  calc = ((x % 16) // 4) * 16 + (x // 16) * 4
  if wants_v_idx == 0:
    print(f"{x:2d}:  {wants_t:2d} {calc:2d}")

 0:   0  0
 4:  16 16
 8:  32 32
12:  48 48
16:   4  4
20:  20 20
24:  36 36
28:  52 52
32:   8  8
36:  24 24
40:  40 40
44:  56 56
48:  12 12
52:  28 28
56:  44 44
60:  60 60


In [ ]:
a = torch.arange(16*16).reshape(16, 16)
# This actually overflows memory, so we need to make the output tensor larger
a_t = torch.zeros(20, 20)

for x in range(64):
    m_idx = x % 16
    n_idx = 4 * (x // 16)
    reg = a[m_idx, n_idx:n_idx+4]
    print(f"{x:2d}: a[{m_idx:2d},{n_idx:2d}:{n_idx+4:<2d}] = {reg}")

    # bad miscompile
    a_t[n_idx, m_idx:m_idx+4] = reg

    print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))



 0: a[ 0, 0:4 ] = tensor([0, 1, 2, 3])
[20x20], torch.float32:
 0  1  2  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0  0  0  0  0  0  

In [28]:
print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))

[20x20], torch.float32:
   0   16   32   48   64   80   96  112  128  144  160  176  192  208  224  240  241  242  243    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   4   20   36   52   68   84  100  116  132  148  164  180  196  212  228  244  245  246  247    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   8   24   40   56   72   88  104  120  136  152  168  184  200  216  232  248  249  250  251    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0  

In [70]:
# This actually overflows memory, so we need to make the output tensor larger
a = torch.zeros(1024, 1024)
a[:16, :16] = torch.arange(16*16).reshape(16, 16)
a_t = torch.zeros(1024, 1024)

for x in range(64):
    for y in range(1):
        m_idx = x
        n_idx = y * 4
        reg = a[m_idx, n_idx:n_idx+4]
        # print(f"{x:2d},{y:2d}: a[{m_idx:2d},{n_idx:2d}:{n_idx+4:<2d}] = {reg}")

        # bad miscompile
        a_t[n_idx:n_idx+4, m_idx] = reg

        # print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))

print(small_tensor_string(a_t[:16, :16], precision=0, sci_mode=False, row_limit=64, col_limit=360))

[16x16], torch.float32:
   0   16   32   48   64   80   96  112  128  144  160  176  192  208  224  240
   1   17   33   49   65   81   97  113  129  145  161  177  193  209  225  241
   2   18   34   50   66   82   98  114  130  146  162  178  194  210  226  242
   3   19   35   51   67   83   99  115  131  147  163  179  195  211  227  243
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0  

In [95]:
# This actually overflows memory, so we need to make the output tensor larger
a_flat = torch.zeros(2048)
a_flat[:256] = torch.arange(256)
a_t = torch.zeros(64, 128)

for wx in range(2):
    for wy in range(2):
        for x in range(64):
            for y in range(1):
                s1 = 16*wx
                s2 = s1 + x
                s3 = y * 4
                s4 = wy * 16
                s5 = s4 + s3
                m_idx = s2
                n_idx = s5
                start = 16*m_idx+n_idx
                end = start + 4
                reg = a_flat[start:end]
                # print(f"{x:2d}: a[{m_idx:2d},{n_idx:2d}:{n_idx+4:<2d}] = {reg}")

                # bad miscompile
                a_t[n_idx:n_idx+4, m_idx] = reg

                # print(small_tensor_string(a_t, precision=0, sci_mode=False, col_limit=300))

print(small_tensor_string(a_t, precision=0, sci_mode=False, row_limit=64, col_limit=10000))

[64x128], torch.float32:
   0   16   32   48   64   80   96  112  128  144  160  176  192  208  224  240    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0
   1   17   33   49   65   81   97  113  129  145  161  177  193  209  225  241    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0    0 

In [ ]:
workgroup_id_0 = 0
workgroup_id_1 = 0

a = torch.arange(16*16)
a_t = torch.zeros(256)

for x in range(64):
    w0_offset = workgroup_id_0 * 16
    i = x % 16
    x_wave_idx = x % 64
    w1_offset = workgroup_id_1 * 16
    j = w1_offset + (x_wave_idx // 16) * 4
    
    flat_idx_start = 16*i+j
    flat_idx_end = flat_idx_start + 4
    a_reg = a[flat_idx_start:flat_idx_end]

    print(f"{x:2d}: a[{i:2d},{j:2d}:{j+4:<2d}] = {a_reg}")

    # bad miscompile
    at_flat_idx_start = 16*j+i
    at_flat_idx_end = at_flat_idx_start + 4
    a_t[at_flat_idx_start:at_flat_idx_end] = a_reg

print(small_tensor_string(a_t.reshape(16, 16), precision=0, sci_mode=False, row_limit=64, col_limit=10000))

 0: a[ 0, 0:4 ] = tensor([0, 1, 2, 3])
 1: a[ 1, 0:4 ] = tensor([16, 17, 18, 19])
 2: a[ 2, 0:4 ] = tensor([32, 33, 34, 35])
 3: a[ 3, 0:4 ] = tensor([48, 49, 50, 51])
 4: a[ 4, 0:4 ] = tensor([64, 65, 66, 67])
 5: a[ 5, 0:4 ] = tensor([80, 81, 82, 83])
 6: a[ 6, 0:4 ] = tensor([96, 97, 98, 99])
 7: a[ 7, 0:4 ] = tensor([112, 113, 114, 115])
 8: a[ 8, 0:4 ] = tensor([128, 129, 130, 131])
 9: a[ 9, 0:4 ] = tensor([144, 145, 146, 147])
10: a[10, 0:4 ] = tensor([160, 161, 162, 163])
11: a[11, 0:4 ] = tensor([176, 177, 178, 179])
12: a[12, 0:4 ] = tensor([192, 193, 194, 195])
13: a[13, 0:4 ] = tensor([208, 209, 210, 211])
14: a[14, 0:4 ] = tensor([224, 225, 226, 227])
15: a[15, 0:4 ] = tensor([240, 241, 242, 243])
16: a[ 0, 4:8 ] = tensor([4, 5, 6, 7])
17: a[ 1, 4:8 ] = tensor([20, 21, 22, 23])
18: a[ 2, 4:8 ] = tensor([36, 37, 38, 39])
19: a[ 3, 4:8 ] = tensor([52, 53, 54, 55])
20: a[ 4, 4:8 ] = tensor([68, 69, 70, 71])
21: a[ 5, 4:8 ] = tensor([84, 85, 86, 87])
22: a[ 6, 4:8 ] = tensor([

In [133]:
workgroup_id_0 = 0
workgroup_id_1 = 0

a = torch.arange(16*16).reshape(16, 16).contiguous()
a_flat = a.flatten()
a_t_ref = a.transpose(0, 1).contiguous()
a_t_ref_flat = a_t.flatten()

a_t_out = torch.zeros(16, 16).contiguous()
a_t_out_flat = a_t_out.view(16*16)

vals_by_thread = [[] for x in range(64)]
threads_by_val = {}

del j, i, x, x_wave_idx, w1_offset, flat_idx_start, flat_idx_end, a_reg

for x in range(64):
    w0_offset = workgroup_id_0 * 16
    i = x % 16
    x_wave_idx = x % 64
    w1_offset = workgroup_id_1 * 16
    j = w1_offset + (x_wave_idx // 16) * 4
    
    flat_idx_start = 16*i+j
    flat_idx_end = flat_idx_start + 4
    a_reg = a_flat[flat_idx_start:flat_idx_end].tolist()
    vals_by_thread[x] = a_reg

    for pos, val in enumerate(a_reg):
        assert val not in threads_by_val
        threads_by_val[val] = (x, pos)

    want = a_t_ref[i, j:j+4].tolist()

    a_reg_str = ', '.join([f"{v:3d}" for v in a_reg])
    want_str = ', '.join([f"{v:3d}" for v in want])

    print(f"{x:2d}: a[{i:2d},{j:2d}:{j+4:<2d}] = {a_reg_str}     "
          f"want a_t[{j:2d},{i:2d}:{i+4:<2d}] = {want_str}")


 0: a[ 0, 0:4 ] =   0,   1,   2,   3     want a_t[ 0, 0:4 ] =   0,  16,  32,  48
 1: a[ 1, 0:4 ] =  16,  17,  18,  19     want a_t[ 0, 1:5 ] =   1,  17,  33,  49
 2: a[ 2, 0:4 ] =  32,  33,  34,  35     want a_t[ 0, 2:6 ] =   2,  18,  34,  50
 3: a[ 3, 0:4 ] =  48,  49,  50,  51     want a_t[ 0, 3:7 ] =   3,  19,  35,  51
 4: a[ 4, 0:4 ] =  64,  65,  66,  67     want a_t[ 0, 4:8 ] =   4,  20,  36,  52
 5: a[ 5, 0:4 ] =  80,  81,  82,  83     want a_t[ 0, 5:9 ] =   5,  21,  37,  53
 6: a[ 6, 0:4 ] =  96,  97,  98,  99     want a_t[ 0, 6:10] =   6,  22,  38,  54
 7: a[ 7, 0:4 ] = 112, 113, 114, 115     want a_t[ 0, 7:11] =   7,  23,  39,  55
 8: a[ 8, 0:4 ] = 128, 129, 130, 131     want a_t[ 0, 8:12] =   8,  24,  40,  56
 9: a[ 9, 0:4 ] = 144, 145, 146, 147     want a_t[ 0, 9:13] =   9,  25,  41,  57
10: a[10, 0:4 ] = 160, 161, 162, 163     want a_t[ 0,10:14] =  10,  26,  42,  58
11: a[11, 0:4 ] = 176, 177, 178, 179     want a_t[ 0,11:15] =  11,  27,  43,  59
12: a[12, 0:4 ] = 192, 193, 

In [ ]:
els_per_thread = 4
threads_per_wave = 64

for x in range(threads_per_wave):
    w0_offset = workgroup_id_0 * 16
    i = x % 16
    x_wave_idx = x % 64
    w1_offset = workgroup_id_1 * 16
    j = w1_offset + (x_wave_idx // 16) * 4

    want_ref = a_t_ref[i, j:j+4].tolist()

    want_idx_ref = []
    for val in want_ref:
        src_thread, src_pos = threads_by_val[val]
        want_idx_ref.append((src_thread, src_pos))


    want_idx = [None]*els_per_thread

    # for shuffle_i in range(els_per_thread):
    #     src_thread = (x+shuffle_i) % els_per_thread * (64 // els_per_thread) + (x // els_per_thread)

    #     src_thread = (x // els_per_thread) * 16 + shuffle_i

    #     src_el = ((els_per_thread - shuffle_i) + (src_thread // (64 // els_per_thread))) % els_per_thread
    #     dest_el_i = (x + shuffle_i) % els_per_thread

    #     want_idx[dest_el_i] = (src_thread, src_el)


    # for dest_i in range(els_per_thread):
    #     src_thread = (x+dest_i) % els_per_thread * (threads_per_wave // els_per_thread) + (x // els_per_thread)

    #     src_thread = ((x // els_per_thread) * 16 + dest_i) % threads_per_wave + els_per_thread * (x // 16)
    #     src_i = (x % 4)

    #     want_idx[dest_i] = (src_thread, src_i)

    for shuff_i in range(els_per_thread):
        dest_offset = (x + shuff_i) % els_per_thread

        src_thread = ((x // els_per_thread) * 16 + dest_offset) % threads_per_wave + els_per_thread * (x // 16)
        src_offset = (x % 4)

        want_idx[dest_offset] = (src_thread, src_offset)
    
    # print(f"{x:2d}: {want_idx_ref[0][0]}")
    # assert want_idx == want_idx_ref
    print(f"{x:2d}: {want_idx_ref}, {want_idx}")


 0: [(0, 0), (1, 0), (2, 0), (3, 0)], [(0, 0), (1, 0), (2, 0), (3, 0)]
 1: [(0, 1), (1, 1), (2, 1), (3, 1)], [(0, 1), (1, 1), (2, 1), (3, 1)]
 2: [(0, 2), (1, 2), (2, 2), (3, 2)], [(0, 2), (1, 2), (2, 2), (3, 2)]
 3: [(0, 3), (1, 3), (2, 3), (3, 3)], [(0, 3), (1, 3), (2, 3), (3, 3)]
 4: [(16, 0), (17, 0), (18, 0), (19, 0)], [(16, 0), (17, 0), (18, 0), (19, 0)]
 5: [(16, 1), (17, 1), (18, 1), (19, 1)], [(16, 1), (17, 1), (18, 1), (19, 1)]
 6: [(16, 2), (17, 2), (18, 2), (19, 2)], [(16, 2), (17, 2), (18, 2), (19, 2)]
 7: [(16, 3), (17, 3), (18, 3), (19, 3)], [(16, 3), (17, 3), (18, 3), (19, 3)]
 8: [(32, 0), (33, 0), (34, 0), (35, 0)], [(32, 0), (33, 0), (34, 0), (35, 0)]
 9: [(32, 1), (33, 1), (34, 1), (35, 1)], [(32, 1), (33, 1), (34, 1), (35, 1)]
10: [(32, 2), (33, 2), (34, 2), (35, 2)], [(32, 2), (33, 2), (34, 2), (35, 2)]
11: [(32, 3), (33, 3), (34, 3), (35, 3)], [(32, 3), (33, 3), (34, 3), (35, 3)]
12: [(48, 0), (49, 0), (50, 0), (51, 0)], [(48, 0), (49, 0), (50, 0), (51, 0)]
13: [

In [ ]:
els_per_thread = 4
threads_per_wave = 64

a_t_out = torch.zeros(16, 16)

for shuff_i in range(els_per_thread):
    print(f"\n\nRound {shuff_i}")
    offers_val = [None for _ in range(threads_per_wave)]
    wants_val = [None for _ in range(threads_per_wave)]
    offers_offset = [None for _ in range(threads_per_wave)]
    wanted_val = [None for _ in range(threads_per_wave)]
    for x in range(threads_per_wave):
        w0_offset = workgroup_id_0 * 16
        i = x % 16
        x_wave_idx = x % 64
        w1_offset = workgroup_id_1 * 16
        j = w1_offset + (x_wave_idx // 16) * 4

        flat_idx_start = 16*i+j
        flat_idx_end = flat_idx_start + 4
        a_reg = a_flat[flat_idx_start:flat_idx_end].tolist()

        want_ref = a_t_ref[i, j:j+4].tolist()

        dest_offset = (x + shuff_i) % els_per_thread

        src_thread = ((x // els_per_thread) * 16 + dest_offset) % threads_per_wave + els_per_thread * (x // 16)
        src_offset = (x % 4)
        offer_offset = (x + els_per_thread - shuff_i) % els_per_thread

        assert offers_val[x] is None
        offers_val[x] = a_reg[offer_offset]
        
        assert offers_offset[x] is None
        offers_offset[x] = offer_offset

        # each thread should get a value and give a value
        assert wants_val[src_thread] is None
        wants_val[src_thread] = want_ref[dest_offset]

        assert wanted_val[src_thread] is None
        wanted_val[src_thread] = src_offset

    for x in range(threads_per_wave):
        w0_offset = workgroup_id_0 * 16
        i = x % 16
        x_wave_idx = x % 64
        w1_offset = workgroup_id_1 * 16
        j = w1_offset + (x_wave_idx // 16) * 4

        flat_idx_start = 16*i+j
        flat_idx_end = flat_idx_start + 4
        a_reg = a_flat[flat_idx_start:flat_idx_end].tolist()

        want_ref = a_t_ref[i, j:j+4].tolist()

        dest_offset = (x + shuff_i) % els_per_thread

        src_thread = ((x // els_per_thread) * 16 + dest_offset) % threads_per_wave + els_per_thread * (x // 16)
        src_offset = (x % 4)
        offer_offset = (x + els_per_thread - shuff_i) % els_per_thread

        a_t_out[i, j+dest_offset] = offers_val[src_thread]

    print("Thread idx", ' '.join([f"{i:3d}" for i in range(64)]))
    print("Offers    ", ' '.join([f"{v:3d}" for v in offers_val]))
    print("Wanted    ", ' '.join([f"{v:3d}" for v in wants_val]))
    print("Offers idx", ' '.join([f"{v:3d}" for v in offers_offset]))
    print("Wants idx ", ' '.join([f"{v:3d}" for v in wanted_val]))

    assert offers_val == wants_val
    assert offers_offset == wanted_val

print(small_tensor_string(a_t_ref, precision=0, sci_mode=False))
print(small_tensor_string(a_t_out, precision=0, sci_mode=False))



Round 0
Thread idx   0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63
Offers       0  17  34  51  64  81  98 115 128 145 162 179 192 209 226 243   4  21  38  55  68  85 102 119 132 149 166 183 196 213 230 247   8  25  42  59  72  89 106 123 136 153 170 187 200 217 234 251  12  29  46  63  76  93 110 127 140 157 174 191 204 221 238 255
Wanted       0  17  34  51  64  81  98 115 128 145 162 179 192 209 226 243   4  21  38  55  68  85 102 119 132 149 166 183 196 213 230 247   8  25  42  59  72  89 106 123 136 153 170 187 200 217 234 251  12  29  46  63  76  93 110 127 140 157 174 191 204 221 238 255
Offers idx   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   0   1   2   3   

In [1]:
import torch

In [30]:
idx_width = 2

def format_thread_idx(t):
    return f"{t[0]:{idx_width}},{t[1]:{idx_width}}"

width = len(format_thread_idx((0, 0)))

In [14]:
dim_m = 16
dim_n = 16
threads_per_wave = 64
els_per_wave = dim_m * dim_n
width = len(str(els_per_wave))

els_per_thread = els_per_wave // threads_per_wave

threads_per_wave_y = dim_n // els_per_thread
threads_per_wave_x = threads_per_wave // threads_per_wave_y

assert threads_per_wave_x * threads_per_wave_y == threads_per_wave
assert threads_per_wave_y * els_per_thread == dim_n

a = torch.arange(dim_m*dim_n, dtype=torch.int32).reshape(dim_m, dim_n)
a_t = a.transpose(0, 1)

vals_by_thread = {}

for x in range(threads_per_wave_x):
    for y in range(threads_per_wave_y):
        row = x
        col = els_per_thread * y

        a_reg = a[row, col:col+els_per_thread].tolist()

        for i, val in enumerate(a_reg):
            vals_by_thread[val] = ((x, y), i)


for x in range(threads_per_wave_x):
    for y in range(threads_per_wave_y):
        row = x
        col = els_per_thread * y
        wants_val = a_t[row, col:col+els_per_thread].tolist()

        wants_thread = [vals_by_thread[v] for v in wants_val]
        wants_thread_str = '   '.join([f"({format_thread_idx(idx)})[{offset}]" for idx, offset in wants_thread])

        print(f"{x},{y}    {wants_thread_str}")

0,0    ( 0, 0)[0]   ( 1, 0)[0]   ( 2, 0)[0]   ( 3, 0)[0]
0,1    ( 4, 0)[0]   ( 5, 0)[0]   ( 6, 0)[0]   ( 7, 0)[0]
0,2    ( 8, 0)[0]   ( 9, 0)[0]   (10, 0)[0]   (11, 0)[0]
0,3    (12, 0)[0]   (13, 0)[0]   (14, 0)[0]   (15, 0)[0]
1,0    ( 0, 0)[1]   ( 1, 0)[1]   ( 2, 0)[1]   ( 3, 0)[1]
1,1    ( 4, 0)[1]   ( 5, 0)[1]   ( 6, 0)[1]   ( 7, 0)[1]
1,2    ( 8, 0)[1]   ( 9, 0)[1]   (10, 0)[1]   (11, 0)[1]
1,3    (12, 0)[1]   (13, 0)[1]   (14, 0)[1]   (15, 0)[1]
2,0    ( 0, 0)[2]   ( 1, 0)[2]   ( 2, 0)[2]   ( 3, 0)[2]
2,1    ( 4, 0)[2]   ( 5, 0)[2]   ( 6, 0)[2]   ( 7, 0)[2]
2,2    ( 8, 0)[2]   ( 9, 0)[2]   (10, 0)[2]   (11, 0)[2]
2,3    (12, 0)[2]   (13, 0)[2]   (14, 0)[2]   (15, 0)[2]
3,0    ( 0, 0)[3]   ( 1, 0)[3]   ( 2, 0)[3]   ( 3, 0)[3]
3,1    ( 4, 0)[3]   ( 5, 0)[3]   ( 6, 0)[3]   ( 7, 0)[3]
3,2    ( 8, 0)[3]   ( 9, 0)[3]   (10, 0)[3]   (11, 0)[3]
3,3    (12, 0)[3]   (13, 0)[3]   (14, 0)[3]   (15, 0)[3]
4,0    ( 0, 1)[0]   ( 1, 1)[0]   ( 2, 1)[0]   ( 3, 1)[0]
4,1    ( 4, 1)[0]   ( 5, 1)[0] 

In [52]:
dim_m = 16
dim_n = 16
threads_per_wave = 64
els_per_wave = dim_m * dim_n

els_per_thread = els_per_wave // threads_per_wave

threads_per_wave_y = dim_n // els_per_thread
threads_per_wave_x = threads_per_wave // threads_per_wave_y

assert threads_per_wave_x * threads_per_wave_y == threads_per_wave
assert threads_per_wave_y * els_per_thread == dim_n

a = torch.arange(dim_m*dim_n, dtype=torch.int32).reshape(dim_m, dim_n)
thread_idxs = [(x, y) for x in range(threads_per_wave_x) for y in range(threads_per_wave_y)]


def compute_shuffle():
    a_t = a.transpose(0, 1)

    for shuff_i in range(els_per_thread):
        print(f"\n\nRound {shuff_i}")
        offers_val = {}
        offers_offset = {}
        wants_val = {}
        wants_thread = {}
        wants_offset = {}
        dests_offset = {}
        wanted_val =  {}
        wanted_offset = {}
        src_threads_x_offset = {}
        for my_idx in thread_idxs:
            x, y = my_idx
            row = x
            col = els_per_thread * y

            dest_offset = (x + shuff_i) % els_per_thread
            offer_offset = (x + els_per_thread - shuff_i) % els_per_thread

            offer_val = a[row, col+offer_offset].item()
            want_val = a_t[row, col+dest_offset].item()


            src_thread, src_offset = vals_by_thread[want_val]

            base_src_thread_x = els_per_thread*y
            src_thread_offset = src_thread[0]-base_src_thread_x

            assert my_idx not in src_threads_x_offset
            src_threads_x_offset[my_idx] = src_thread_offset

            assert my_idx not in offers_val
            offers_val[my_idx] = offer_val
            
            assert my_idx not in offers_offset
            offers_offset[my_idx] = offer_offset


            wants_thread[my_idx] = src_thread
            wants_offset[my_idx] = src_offset
            wants_val[my_idx] = want_val
            dests_offset[my_idx] = dest_offset

            # each thread should get a value and give a value
            assert src_thread not in wanted_val
            wanted_val[src_thread] = want_val

            assert src_thread not in wanted_offset
            wanted_offset[src_thread] = src_offset

        print("Thread idx            ", '  '.join([f"{format_thread_idx(t):{width}}" for t in thread_idxs]))
        print("Wants val             ", '  '.join([f"{wants_val[idx]:{width}d}" for idx in thread_idxs]))
        print("Wants from thread     ", '  '.join([f"{format_thread_idx(wants_thread[idx]):{width}}" for idx in thread_idxs]))
        print("Wants thread x offset ", '  '.join([f"{src_threads_x_offset[idx]:{width}d}" for idx in thread_idxs]))
        print("Wants offset          ", '  '.join([f"{wants_offset[idx]:{width}d}" for idx in thread_idxs]))
        print("Dests offset          ", '  '.join([f"{dests_offset[idx]:{width}d}" for idx in thread_idxs]))
        print("Wanted val            ", '  '.join([f"{wanted_val[idx]:{width}d}" for idx in thread_idxs]))
        print("Offers val            ", '  '.join([f"{offers_val[idx]:{width}d}" for idx in thread_idxs]))
        print("Wanted offset         ", '  '.join([f"{wanted_offset[idx]:{width}d}" for idx in thread_idxs]))
        print("Offers offset         ", '  '.join([f"{offers_offset[idx]:{width}d}" for idx in thread_idxs]))

        assert offers_val == wanted_val
        assert offers_offset == wanted_offset
        assert dests_offset == src_threads_x_offset

compute_shuffle()



Round 0
Thread idx              0, 0   0, 1   0, 2   0, 3   1, 0   1, 1   1, 2   1, 3   2, 0   2, 1   2, 2   2, 3   3, 0   3, 1   3, 2   3, 3   4, 0   4, 1   4, 2   4, 3   5, 0   5, 1   5, 2   5, 3   6, 0   6, 1   6, 2   6, 3   7, 0   7, 1   7, 2   7, 3   8, 0   8, 1   8, 2   8, 3   9, 0   9, 1   9, 2   9, 3  10, 0  10, 1  10, 2  10, 3  11, 0  11, 1  11, 2  11, 3  12, 0  12, 1  12, 2  12, 3  13, 0  13, 1  13, 2  13, 3  14, 0  14, 1  14, 2  14, 3  15, 0  15, 1  15, 2  15, 3
Wants val                  0     64    128    192     17     81    145    209     34     98    162    226     51    115    179    243      4     68    132    196     21     85    149    213     38    102    166    230     55    119    183    247      8     72    136    200     25     89    153    217     42    106    170    234     59    123    187    251     12     76    140    204     29     93    157    221     46    110    174    238     63    127    191    255
Wants from thread       0, 0   4, 0   8, 0  12, 0 

In [72]:
def simulate_shuffle():
    a_t_out = torch.zeros_like(a_t)

    a_regs = {}
    a_t_regs = {}
    for my_idx in thread_idxs:
        x, y = my_idx
        row = x
        col = els_per_thread * y

        a_regs[my_idx] = a[row, col:col+els_per_thread]
        a_t_regs[my_idx] = torch.zeros(4)

    for shuff_i in range(els_per_thread):
        offers = {}

        for my_idx in thread_idxs:
            x, y = my_idx
            offer_offset = (x + els_per_thread - shuff_i) % els_per_thread

            offers[my_idx] = a_regs[my_idx][offer_offset]


        for my_idx in thread_idxs:
            x, y = my_idx
            dest_offset = (x + shuff_i) % els_per_thread

            src_thread = els_per_thread*y + dest_offset, x // 4
            src_thread_linear = 4*src_thread[0] + src_thread[1]
            assert src_thread_linear < threads_per_wave

            shuffle_val = offers[src_thread]

            a_t_regs[my_idx][dest_offset] = shuffle_val

        for my_idx in thread_idxs:
            x, y = my_idx
            x, y = my_idx
            row = x
            col = els_per_thread * y

            a_t_out[row, col:col+els_per_thread] = a_t_regs[my_idx]

    assert_close(a_t_out, a.transpose(0, 1))
    
simulate_shuffle()